# LeNet-5 Neural Network for MNIST

### Options

In [18]:
batch_size = 20
training_epochs = 5

### Load & convert data

In [19]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

def convert_data(x_data: np.ndarray, y_data: np.ndarray):
    # hacky way to increase the size of a (unknown)x28x28 to (unknown)x32x32
    # by appending two zeros at all sides of the 28x28 matrix
    tmp = np.zeros((x_data.shape[0], 2, x_data.shape[2]))
    x_data = np.append(x_data, tmp, axis=1)
    x_data = np.append(tmp, x_data, axis=1)
    tmp = np.zeros((x_data.shape[0], x_data.shape[1], 2))
    x_data = np.append(x_data, tmp, axis=2)
    x_data = np.append(tmp, x_data, axis=2)

    # change dimensions: (unknown)x32x32 to (unknown)x32x32x1
    x_data = np.expand_dims(x_data, -1)
    # change dimensions: (unknown) to (unknown)x10
    y_data = keras.utils.to_categorical(y_data, 10)

    # return (x_data - 128, y_data)
    return (x_data / 256.0, y_data)


# for now we are just interested in the training data and not the test data
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

(X_train, y_train) = convert_data(X_train, y_train)
(X_test, y_test) = convert_data(X_test, y_test)

In [ ]:
# save data
np.save("export/mnist/x_train.npy", X_train)
np.save("export/mnist/y_train.npy", y_train)
np.save("export/mnist/x_test.npy", X_test)
np.save("export/mnist/y_test.npy", y_test)

In [ ]:
print("Shapes:")
print((X_train.shape, y_train.shape))

Shapes:
((60000, 32, 32, 1), (60000, 10))


### Setup our model

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(32, 32, 1)),
    keras.layers.Conv2D(6, kernel_size=(5, 5), activation="relu"),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Conv2D(16, kernel_size=(5, 5), activation="relu"),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(120),
    keras.layers.Dense(84),
    keras.layers.Dense(10),
    keras.layers.Activation('softmax')
])

model.summary()

2024-11-26 23:42:23.088657: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 6)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 10, 10, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 120)            │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           850 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 10)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,706 (241.04 KB)

 Trainable params: 61,706 (241.04 KB)

 Non-trainable params: 0 (0.00 B)

### Fitting the model

In [ ]:
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

hist = model.fit(X_train, y_train, batch_size=batch_size, epochs=training_epochs, validation_split=0.2)

Epoch 1/20


2024-11-26 23:42:23.926886: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 196608000 exceeds 10% of free system memory.


2400/2400 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.8990 - loss: 0.3202 - val_accuracy: 0.9811 - val_loss: 0.0611
Epoch 2/20
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.9807 - loss: 0.0641 - val_accuracy: 0.9827 - val_loss: 0.0588
Epoch 3/20
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.9849 - loss: 0.0464 - val_accuracy: 0.9865 - val_loss: 0.0493
Epoch 4/20
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.9877 - loss: 0.0380 - val_accuracy: 0.9785 - val_loss: 0.0715
Epoch 5/20
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.9891 - loss: 0.0343 - val_accuracy: 0.9859 - val_loss: 0.0526
Epoch 6/20
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.9900 - loss: 0.0300 - val_accuracy: 0.9845 - val_loss: 0.0574
Epoch 7/20
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.9912 - loss: 0.0285 - val_accuracy: 0.9856 - val_loss: 0.0560
Epoch 8/20
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.9913 - loss: 0.0258 - val_accu

## How good is our model for non-train data?

In [ ]:
score = model.evaluate(X_test, y_test, batch_size=batch_size)

score

500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9889 - loss: 0.0530


[0.039324551820755005, 0.9908999800682068]

In [ ]:
model.predict(X_train[3:5])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step


array([[4.28403813e-13, 1.00000000e+00, 4.75918003e-12, 6.18441466e-16,
        1.27518023e-08, 7.56549023e-16, 4.50301428e-12, 4.28326846e-10,
        4.08337079e-12, 2.39461584e-14],
       [4.97525150e-17, 1.79832697e-15, 1.06621138e-19, 2.11866313e-16,
        2.85140436e-11, 1.10588506e-17, 8.60469199e-26, 1.87558760e-14,
        5.56502645e-14, 1.00000000e+00]], dtype=float32)

In [ ]:
from pathlib import Path

Path("export/").mkdir(exist_ok=True)
model.save("export/original_model.keras")

In [ ]:
model.export("export/tmp")

INFO:tensorflow:Assets written to: export/tmp/assets


INFO:tensorflow:Assets written to: export/tmp/assets


Saved artifact at 'export/tmp'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32, 32, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  132209108406480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132209108405520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132209108407440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132209108408400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132209108408016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132209108408592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132209108405136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132209108408976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132209108407824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132209108409552: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [ ]:
!python -m tf2onnx.convert --saved-model export/tmp --output export/original_model.onnx

2024-11-26 23:47:52.291752: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1732661272.310498  172473 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1732661272.315934  172473 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-26 23:47:52.334369: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
<frozen runpy>:128: RuntimeWarning: 'tf2onnx.convert' found in sys.modules after import of package 'tf2onnx', but prior to ex

In [ ]:
# import tensorflow_model_optimization as tfmot

# quantize_model = tfmot.quantization.keras.quantize_model
# q_aware_model = quantize_model(model)

In [ ]:
# q_aware_model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

# q_aware_model.summary()

In [ ]:
# hist = model.fit(X_train, y_train, batch_size=batch_size, epochs=training_epochs, validation_split=0.2)

In [ ]:
# converter = tf.lite.TFLiteConverter.from_keras_model(q_aware_model)
# converter.optimizations = [tf.lite.Optimize.DEFAULT]

# tflite_qaware_model = converter.convert()


In [ ]:
# with open("export/quantization_aware_training.tflite", 'wb') as f:
#     f.write(tflite_qaware_model)